# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import logging
import os
import sys
import pandas as pd

from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("..")

from processor.core.interaction_conductor.llm_conductor import LLMConductor
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.core.ir_system.ir_data_model import AbstractDocument
from processor.core.ir_system.ir_data_model import Table, TableContext
from processor.core.ir_system.ir_data_model import Knowledge, convert_multi_retriever_results_to_str
from processor.model.interface.model_factory import get_embed_model, get_llm
from processor.model.llm_message import Role
from processor.model.option import LLMOption
from processor.utils.json_processor import parse_json

from logging import Logger
from pandas import DataFrame

logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)

# LLMConductor

## Definition

In [ ]:
class Interaction:
    """
    Basically keeps track of every call to the process_input() function of LLMConductor
    """
    def __init__(self, human_input: str, llm_response: str) -> None:
        self.human_input = human_input
        self.llm_response = llm_response

    def __str__(self) -> str:
        return f"""{{"human input": {self.human_input}, "llm response": {self.llm_response}}}"""

class InformationNeedState:
    """
    Represents user's information need as a set of target schemas and SQLs to be executed over them.
    For example, if the user needs to know about the work addresses of faculty members, the target schemas
    may be ["name", "work address"], where name represents the names of the members, and work address represents
    the corresponding work address of each of them. After materialized by Materializer Engine, the SQLs can be
    executed sequentially over the materialized tables, and the outcome is useful to answer user's needs.
    """
    def __init__(self) -> None:
        self.target_schemas: dict[str, DataFrame] = dict()
        self.is_target_schemas_materialized = False
        self.column_descriptions: dict[str, dict[str, str]] = dict()
        self.sqls: list[str] = []

    def __str__(self) -> str:
        target_schemas_repr = ""
        for schema_id in self.target_schemas:
            table = self.target_schemas[schema_id]
            target_schemas_repr += f"\n- Table {schema_id}:\ncol: {" | ".join(list(table.columns))}"
            if len(table) > 0:
                # Sample 5 rows to represent the table
                sample_rows = table.sample(min(5, len(table)), random_state=42)
                sample_row_idx = 1
                for _, data in sample_rows.iterrows():
                    str_data = [str(i) for i in data]
                    target_schemas_repr += (
                        f"\n- sample row {sample_row_idx}: {" | ".join(str_data)}"
                    )
                    sample_row_idx += 1
            target_schemas_repr += "\n"
        return f"""Target schemas (Is materialized? {self.is_target_schemas_materialized}):
{target_schemas_repr.strip()}

Column descriptions of target schemas:
{self.column_descriptions}

SQLs to be run sequentially over the target schemas:
{self.sqls}"""

In [ ]:
from processor.model.llm_message import LLMMessage


class ICPromptFactory:
    def get_sys_prompt(self, iteration_limit: int) -> str:
      return f"""Your role is to guide users in detecting, clarifying, and formalizing their possibly ambiguous information needs, eventually fulfilling them through structured data operations. You must converse and collaborate with users in evolving an Information Need State, which reflects their underlying information needs. This is a structured representation, consisting of:
    - `target_schemas` (dict[str, list[str]): A set of table schemas relevant to what users are looking for. The format is as follows: {{"Schema_ID_1": ["col_1", …], "Schema_ID_2": …, …}}. Each schema ID represents a conceptually coherent table. Each table is relevant to users' information needs. Target schemas, after finalized (i.e., confirmed with users), can be materialized by an external tool (more about this later).
    - `column_descriptions` (dict[str, dict[str, str]]): The descriptions of the columns of all target schemas. The format is as follows: {{"Schema_ID_1": {{"col_1": "This column represents …"}}, …}}
    - `sqls` (list[str]): A list of SQL queries over the (materialized) target schemas. Executing them (by an external tool) sequentially should produce relevant information to satisfy the information needs of the users.
The end-to-and process is called a session, which is specific to a user. In each session, Information Need State starts empty but evolves over the course of the session. You must ensure the process is transparent and collaborative.

## Workflow
A session consists of multiple back-and-forth steps. In each step, you have at most {iteration_limit} iterations to select any of the following actions (mutually exclusive):
    - `internal_reasoning`: Reflect out loud (for yourself only).
    - `tool_call`: Call a tool to retrieve relevant information, evolve the state, etc.
    - `communicate_with_user`: Produce a user-facing message, which is either a summary of your actions in the step or a clarifying question.
Remember to close a step with `communicate_with_user`, so that they are aware of what has been done.

Some principles to remember:
- You CANNOT mix tool_call with internal_reasoning or communicate_with_user.
- The current state represents your current best understanding of the user needs. It may not represent what the user actually wants at the end, but you can materialize it and run sqls on it if necessary. This is useful, for instance, to ground your understanding and help guide and inform users.
- If you want to showcase or refer to some documents you retrieved from the IR system, you can mention their IDs in the message of your `communicate_with_user` action, since the user can inspect them when interacting with you.
---

## AVAILABLE TOOLS
- **IR System**
    - Retrieves relevant tabular or textual data from our database based on natural-language prompts
    - For general inquiries, you may not need to use this tool and rely on your knowledge, but state clearly the sources of your information in the user-facing message.
    - Use when new or updated data is needed, but remember that calling this tool erases previously retrieved data (if any).
    - Args: `{{"prompt": "<retrieval query>"}}`

- **State Manipulation**
    - Updates Information Need State
    - Use when you have gathered enough signal to represent part of the user's needs formally. If user disagrees, iterate.
    - If the conversation has gone off-course, you can always reset the state (setting the values of target_schemas, column_descriptions, and sqls to be empty) and collaboratively rebuilding it with the user.
    - Users may inspect and give feedback on the current state at any time
    - You can modify only the target schemas (and column descriptions) or only the sqls. Just set what you do not want to change to be null.
    - Args (choose any of the following, which represent modifying all, only target schemas (and column descriptions), or only sqls, respectively): 
        {{
        "target_schemas": {{ "<id>": [<list of descriptive column names>] }},
        "column_descriptions": null | {{"<id>": {{ "<column name>": "<description of the column>" }} }}
        "sqls": [<list of SQL strings over target schema IDs>]
        }}

        {{
        "target_schemas": {{ "<id>": [<list of descriptive column names>] }},
        "column_descriptions": null | {{"<id>": {{ "<column name>": "<description of the column>" }} }}
        }}

        {{
        "sqls": [<list of SQL strings over target schema IDs>]
        }}

- **Materializer Engine**
    - Fills the current target schemas with actual data
    - Args: `""` (no input).
    - **VERY IMPORTANT**: DO NOT be too eager to call Materializer Engine, especially when the user needs is still a bit general/exploratory/vague. This is a costly operation.

- **SQL Engine**
  - Runs the SQLs in the state on the materialized target schemas (you can call this only after the target schemas have been materialized). 
  - Args: `""` (no input).
  - Again, remember to ensure that if you want to execute the SQLs, ensure that the target schemas have been materialized, else you will get empty result or errors."""

    def get_env_state_prompt(
        self,
        curr_iteration: int,
        max_iteration: int,
        info_need_state: InformationNeedState,
        interaction_history: list[Interaction],
        actions_taken: list[str],
        curr_retrieval_results: dict[RetrieverType, list[AbstractDocument]],
        human_input: str,
    ) -> str:
        return f"""Relevant information for the current iteration in this step (iteration {curr_iteration} out of {max_iteration}):

INFORMATION NEED STATE:
{info_need_state}

ACTIONS YOU HAVE TAKEN FROM PREVIOUS ITERATIONS IN THIS STEP:
{actions_taken}

INTERACTION HISTORY (PAIRS OF HUMAN INPUT AND YOUR HUMAN-FACING RESPONSE):
{self.__convert_interactions_to_str(interaction_history)}

PREVIOUSLY RETRIEVED DATA FROM THE IR SYSTEM:
{convert_multi_retriever_results_to_str(curr_retrieval_results)}

CURRENT HUMAN INPUT:
{human_input}

Please output your decision for this step in either of the following formats (depending on intent):
{{
    "intent": "communicate_with_user" | "internal_reasoning",
    "message": "<string if intent is communicate_with_user or internal_reasoning>"
}}

{{
    "intent": "tool_call",
    "tool": "IR System" | "Materializer Engine" | "State Manipulation" | "SQL Engine",
    "args": { ... }
}}

- **VERY IMPORTANT NOTE**: Again, DO NOT be too eager to call Materializer Engine, especially when the user needs is still general/exploratory/vague. This is a costly operation."""
    
    def get_knowledge_extraction_prompt(
    self,
    human_input: str
) -> str:
        return f"""You are very talented in inferring knowledge from a text.
You are given a human input to a question-answering system: ```{human_input}```
Please consider whether it consists domain knowledge that will be helpful for other people using the system. Make sure you only extract general knowledge that does not just apply to a specific user. If there is none, then do not force for there to be any.

When you find multiple pieces of related information, combine them into a single comprehensive knowledge statement rather than splitting them into separate points. The goal is to capture the complete context and relationships in one cohesive statement.

For example, if the input is:
"I need to check if this new purchase order follows our department's policy of requiring at least 3 quotes for purchases over $10,000. The policy also states that these quotes must be from different suppliers and obtained within the last 30 days."

The output would be:
{{
    "contains_domain_knowledge": true,
    "domain_knowledge": [
        "Department purchasing policy requires at least 3 different supplier quotes obtained within 30 days for any purchase over $10,000"
    ]
}}

Please output your decision in the following format:
{{
    "contains_domain_knowledge": true | false,
    "domain_knowledge": null | [<list of domain knowledge strings if any>]
}}"""

    def get_direct_response_anyway_prompt(self) -> str:
        return """You have reached the iteration limit for this step. Please summarize the actions that you have done.
You are essentially asked to produce a `communicate_with_user` response but without the JSON format requirements. Simply output the summary."""   
    
    def __convert_interactions_to_str(self, interactions: list[Interaction]) -> str:
        interaction_repr = ""
        for interaction in interactions:
            interaction_repr += f"- {interaction}\n"
        interaction_repr = interaction_repr.strip()
        return interaction_repr

In [ ]:
from typing import Optional
import duckdb
from processor.core.ir_system.lm_interface import LMInterface
from processor.core.materializer_engine.llm_planner import LLMPlanner


ITERATION_LIMIT = 5
PAST_INTERACTIONS_LIMIT = 5


class LLMConductor:
    def __init__(self, llm_path: str, embed_path: str, logger: Logger) -> None:
        self.llm = get_llm(llm_path)(llm_path)
        self.embed_model = get_embed_model()(embed_path)
        self.logger = logger

        self.info_need_state = InformationNeedState()
        self.interaction_history: list[Interaction] = []

        self.prompt_factory = ICPromptFactory()
        self.current_retrieval_results: dict[RetrieverType, list[AbstractDocument]] = (
            dict()
        )

        self.materializer = LLMPlanner(self.llm, self.logger, self.embed_model)

    def process_input(self, human_input: str, human_id: str) -> str:
        self.logger.info(f"Processing human input: {human_input}")
        # self.logger.info(f"Preliminary step: extracting domain knowledge")
        # domain_knowledge_extraction_messages = [
        #     LLMMessage(
        #         role=Role.SYSTEM.value,
        #         content=self.prompt_factory.get_knowledge_extraction_prompt(
        #             human_input
        #         ),
        #     )
        # ]
        # domain_knowledge_extraction_decision = self.llm.chat(
        #     domain_knowledge_extraction_messages, LLMOption(json_mode=True)
        # )
        # extraction_decision_json = parse_json(domain_knowledge_extraction_decision)
        # if extraction_decision_json["contains_domain_knowledge"]:
        #     domain_knowledge: list[str] = extraction_decision_json["domain_knowledge"]
        #     self.logger.info(f"=> Domain knowledge extracted: {domain_knowledge}")
        #     domain_knowledge_docs: list[AbstractDocument] = [
        #         Knowledge(
        #             doc_id="new_doc",
        #             retriever_type=RetrieverType.KNOWLEDGE_BASE,
        #             content=curr_domain_knowledge,
        #             metadata={"type": "global", "user": human_id},
        #         )
        #         for curr_domain_knowledge in domain_knowledge
        #     ]
        #     ir_system = LMInterface(
        #         {"llm": self.llm, "embed_model": self.embed_model},
        #         self.logger,
        #     )
        #     ir_system.index_documents(
        #         RetrieverType.KNOWLEDGE_BASE, domain_knowledge_docs
        #     )

        num_iteration = 0
        user_facing_response = ""
        is_user_facing_response = False
        llm_messages = [
            LLMMessage(
                role=Role.SYSTEM.value,
                content=self.prompt_factory.get_sys_prompt(ITERATION_LIMIT),
            )
        ]
        actions_taken: list[str] = []
        while not is_user_facing_response and num_iteration < ITERATION_LIMIT:
            num_iteration += 1
            llm_messages.append(
                LLMMessage(
                    role=Role.USER.value,
                    content=self.prompt_factory.get_env_state_prompt(
                        num_iteration,
                        ITERATION_LIMIT,
                        self.info_need_state,
                        self.interaction_history,
                        actions_taken,
                        self.current_retrieval_results,
                        human_input,
                    ),
                )
            )

            llm_output = self.llm.chat(llm_messages, LLMOption(json_mode=True))
            llm_messages.append(
                LLMMessage(role=Role.ASSISTANT.value, content=llm_output)
            )
            """Format of action:
            {
                "intent": "communicate_with_user" | "internal_reasoning" | "tool_call",
                "message": null | "<string>",
                "tool": null | "IR System" | "Materializer Engine" | "State Manipulation" | "SQL Engine",
                "args": null | { ... }
            }
            """
            action = parse_json(llm_output)
            intent: str = action.get("intent")
            action_message: None | str = action.get("message")
            tool: None | str = action.get("tool")
            args: None | dict = action.get("args")

            if intent == "communicate_with_user" and isinstance(action_message, str):
                self.interaction_history.append(
                    Interaction(human_input, action_message)
                )
                user_facing_response = action_message
                is_user_facing_response = True
            elif intent == "internal_reasoning" and isinstance(action_message, str):
                llm_messages.append(
                    LLMMessage(
                        role=Role.USER.value,
                        content=f"You did some internal reasoning: {action_message}",
                    )
                )
            elif (intent == "tool_call" or intent != "communicate_with_user" or intent != "internal_reasoning") and tool is not None:
                tool_outcome = self.__execute_tool(tool, args)
                llm_messages.append(
                    LLMMessage(role=Role.USER.value, content=tool_outcome)
                )

        if not is_user_facing_response:
            self.logger.info("Force produce user-facing response")
            llm_messages.append(
                LLMMessage(
                    role=Role.SYSTEM.value,
                    content=self.prompt_factory.get_direct_response_anyway_prompt(),
                )
            )
            user_facing_response = self.llm.chat(llm_messages)
            self.interaction_history.append(
                Interaction(human_input, user_facing_response)
            )
        return user_facing_response

    def __execute_tool(self, tool: str, args: str | dict) -> str:
        if tool == "IR System" and isinstance(args, dict):
            self.logger.info(f"IR System request with params: {args}")
            ir_system = LMInterface(
                {"llm": self.llm, "embed_model": self.embed_model},
                self.logger,
            )
            self.current_retrieval_results = ir_system.retrieve_documents(
                args["prompt"],
                ["environment"],
                10,  # Future-TODO: Change hard-coded sources and k
            )
            return "Successfully retrieved documents from the IR system. Notice that the `PREVIOUSLY RETRIEVED DATA FROM THE IR SYSTEM` has been updated."
        elif tool == "State Manipulation" and isinstance(args, dict):
            self.logger.info(f"State Manipulation request with params: {args}")
            target_schemas: dict[str, list[str]] | None = args.get("target_schemas")
            column_descriptions: dict[str, dict[str, str]] | None = args.get("column_descriptions")
            sqls: list[str] | None = args.get("sqls")

            modifications_happening = False
            if target_schemas is not None and column_descriptions is not None:
                if column_descriptions is not None:
                    target_schemas_df: dict[str, DataFrame] = dict()
                    for schema_id in target_schemas:
                        target_schemas_df[schema_id] = pd.DataFrame(columns=target_schemas[schema_id])
                    self.info_need_state.target_schemas = target_schemas_df
                    self.info_need_state.column_descriptions = column_descriptions
                    self.info_need_state.is_target_schemas_materialized = False
                    modifications_happening = True
                else:
                    return "If you want to change target_schemas, make sure to also define column_descriptions."

            if sqls is not None:
                self.info_need_state.sqls = sqls
                modifications_happening = True
            
            if modifications_happening:
                return "Successfully modified the state."
            return "No modification is done."
        elif tool == "Materializer Engine":
            self.logger.info(f"Materializer Engine called")
            self.info_need_state.target_schemas = self.materializer.materialize_target_schemas(
                self.info_need_state.target_schemas,
                self.info_need_state.column_descriptions,
                self.info_need_state.sqls,
            )
            self.info_need_state.is_target_schemas_materialized = True
            return "Successfully materialized the target schemas."
        elif tool == "SQL Engine":
            self.logger.info("SQL Engine called")
            execution_result: str = ""

            if isinstance(args, dict) and "custom_sql" in args.keys():
                custom_sql: str = args["custom_sql"]
                execution_result = self.__execute_sqls(custom_sql)
            else:
                if not self.info_need_state.is_target_schemas_materialized:
                    return "Target schemas have not been materialized, so running SQL Engine will produce empty results."
                if len(self.info_need_state.sqls) == 0:
                    return "sqls is still empty, which means there is nothing to execute."
                execution_result = self.__execute_sqls()

            self.logger.info(f"SQL execution result output: {execution_result}")
            return f"Executed the SQLs, which resulted in this output: {execution_result}"
        return "Tool calling failed."

    def __execute_sqls(self, custom_query: Optional[str] = None):
        """
        Executes the SQLs (sequentially) over the target schemas.
        The result (for now) is a scalar (converted to string).
        """
        # Create an in-memory DuckDB connection
        con = duckdb.connect(database=":memory:")

        if custom_query:
            tables: dict[str, DataFrame] = dict()
            tables_only_results = self.current_retrieval_results[RetrieverType.PNEUMA]
            for table in tables_only_results:
                cleaned_doc_id = table.doc_id.split("/")[-1]
                tables[cleaned_doc_id] = table.content
            self.logger.info(
                f"Executing `{custom_query}` on the retrieved tables"
            )
            sqls: list[str] = [custom_query]
        else:
            tables: dict[str, DataFrame] = self.info_need_state.target_schemas
            sqls: list[str] = self.info_need_state.sqls
            self.logger.info(
                f"Executing {sqls} SQL statements on the (materialized) target schemas"
            )

        # Register each table into DuckDB
        for table_name, df in tables.items():
            con.register(table_name, df)

        result = DataFrame()
        for sql in sqls:
            self.logger.info(f"Executing SQL: {sql}")
            result = con.execute(sql).fetchdf()

        self.logger.info(f"Final result shape: {result.shape}")

        # If the result has only one cell, return it as a scalar string
        if result is not None and result.shape == (1, 1):
            return str(result.iat[0, 0])

        return str(result)

## Evaluation Scenario: Environment Dataset

In [ ]:
llm_path = "model/weight/qwen3-8b"
embed_model_path = "model/weight/bge-base"
llm_conductor = LLMConductor(llm_path, embed_model_path, logger)

In [ ]:
INDEXING = False
if INDEXING:
    DATASET_DIR = "../../data_src/environment/dataset"
    dataset_metadata = pd.read_csv("../../data_src/environment/metadata.csv")

    documents: list[AbstractDocument] = []
    dataset = os.listdir(DATASET_DIR)
    for table_name in dataset:
        table = pd.read_csv(f"{DATASET_DIR}/{table_name}")
        documents.append(
            Table(
                doc_id=f"{DATASET_DIR}/{table_name}",
                retriever_type=RetrieverType.PNEUMA,
                content=table,
                metadata={
                    "table_name": f"{DATASET_DIR}/{table_name}",
                    "dataset_name": "environment"
                }
            )
        )
    for idx, row in dataset_metadata.iterrows():
        table_name = row["table_name"]  # TODO: hati2 maslaah table-name karena path nya beda
        description = row["description"]
        documents.append(
            TableContext(
                doc_id=f"context_{DATASET_DIR}/{table_name}",
                retriever_type=RetrieverType.PNEUMA,
                content=description,
                metadata={
                    "table_name": f"{DATASET_DIR}/{table_name}",
                    "dataset_name": "environment",
                    "type": "description",
                }
            )
        )

    ir_sys = LMInterface(
        {"llm": llm_conductor.llm, "embed_model": llm_conductor.embed_model},
        llm_conductor.logger,
    )
    ir_sys.index_documents(
        RetrieverType.PNEUMA,
        documents
    )

In [ ]:
USER_ID = "llm"

In [ ]:
llm_conductor.process_input(
    "I'm curious about beach water quality issues—specifically, how often beaches in the Northeast had to close due to contamination. Can we start by looking at bacterial exceedances for coastal states like Massachusetts, maybe broken down by year? I'm not sure if closures are directly tracked, but exceedances might be a good place to begin.",
    USER_ID,
)

In [ ]:
llm_conductor.info_need_state.target_schemas

In [ ]:
llm_conductor.info_need_state.column_descriptions

In [ ]:
llm_conductor.info_need_state.sqls

In [ ]:
# retrieval_results = llm_conductor.current_retrieval_results[RetrieverType.PNEUMA]
# for result in retrieval_results:
#     print(f"Table Name: {result.doc_id}; Table Columns: {result.content.columns}")

In [ ]:
# llm_conductor.process_input(
#     """Let’s go ahead and focus on the 2013 bathing season in Massachusetts. That’s typically a key year in our historical comparisons, and I’m curious about both the frequency of violations and the number of samples collected.

# Could you show me, just for 2013, how many total water samples were collected across all Massachusetts beaches, and how many of those resulted in bacterial exceedances? If possible, break that down by beach so I can spot any hotspots.""",
#     USER_ID,
# )